# 07 · Backpropagation: every gradient in one backward pass

*Neural Networks from Scratch: Build, Train and Understand AI Models*, notebook 7 of 12

This is the notebook where your network learns how to learn. A numerical gradient needs two forward passes for **every** weight, which is hopeless for real networks. **Backpropagation** gets the exact gradient of the loss with respect to every weight and bias in a single backward pass, at about the cost of one more forward pass. You will derive each piece on paper, type it, and then prove your code right with the gradient checker from notebook 06.

## What you will build

- `softmax_cross_entropy_grad(probs, y)`: the gradient of the loss with respect to the output logits, $(P - Y)/n$;
- `dense_backward(dZ, A_prev, W)`: the gradients of one dense layer, returning `(dW, db, dA_prev)`;
- `relu_backward(dA, Z)`: the gradient through a ReLU;
- `backward(network, cache, probs, y)`: the full backward pass, returning a gradient for every layer.

## Learning objectives

By the end of this notebook you will be able to:

1. derive $\partial L / \partial Z = (P - Y)/n$ for softmax followed by cross-entropy;
2. derive the gradients of a dense layer, $A^\top dZ$, $\sum_i dZ_i$ and $dZ\, W^\top$, and check them with shapes;
3. pass a gradient back through ReLU with a mask;
4. assemble the full backward pass of a multilayer network from these pieces;
5. gradient-check every parameter of a network, explain why the check works, and use it to locate a bug.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for one plot. Nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab. The standard-library `time` module measures one speed comparison.
- Nothing is downloaded, and every random number comes from a seeded generator.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you write against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import matplotlib.pyplot as plt

## 📦 Recap: what you built earlier

The hidden cell below re-defines the pieces from notebooks 01–06 under their usual names: the network (`relu`, `relu_grad`, `sigmoid`, `softmax`, `dense_forward`, `init_network`, `forward`, `predict`, `accuracy`, `make_spirals`), the losses (`mse`, `binary_cross_entropy`, `cross_entropy`, `cross_entropy_from_logits`) and the gradient tools (`numerical_derivative`, `numerical_gradient`, `relative_error`). Run it; open it only if you want to look back at the code.

In [ ]:
# 📦 Recap: the pieces from notebooks 01–04, under their usual names.

def relu(Z):
    """ReLU, element by element: max(0, Z)."""
    return np.maximum(0.0, Z)


def relu_grad(Z):
    """The slope of ReLU: 1.0 where Z > 0, else 0.0 (we take 0 at Z = 0)."""
    return (Z > 0).astype(float)


def sigmoid(Z):
    """Numerically stable logistic function 1 / (1 + exp(-Z)), element by element."""
    return np.exp(-np.logaddexp(0.0, -Z))


def softmax(Z):
    """Row-wise softmax of logits Z (batch, classes). Subtracting each row's maximum avoids overflow."""
    exps = np.exp(Z - Z.max(axis=1, keepdims=True))
    return exps / exps.sum(axis=1, keepdims=True)


def dense_forward(X, W, b):
    """Z = X @ W + b. X: (batch, n_in), W: (n_in, n_out), b: (n_out,) -> Z: (batch, n_out)."""
    return X @ W + b


def init_network(sizes, rng):
    """He-initialised layers for sizes such as [2, 64, 3]: a list of {"W": (n_in, n_out), "b": (n_out,)}."""
    return [
        {"W": rng.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out)), "b": np.zeros(n_out)}
        for n_in, n_out in zip(sizes[:-1], sizes[1:])
    ]


def forward(network, X):
    """ReLU hidden layers and a softmax output.

    Returns (probs, cache): probs has shape (batch, classes), and cache[i] = (A_prev, Z) for layer i.
    """
    cache = []
    A = X
    for index, layer in enumerate(network):
        Z = dense_forward(A, layer["W"], layer["b"])
        cache.append((A, Z))
        A = softmax(Z) if index == len(network) - 1 else relu(Z)
    return A, cache


def predict(network, X):
    """The predicted class index for every row of X: shape (batch,)."""
    probs, _ = forward(network, X)
    return probs.argmax(axis=1)


def accuracy(network, X, y):
    """The fraction of rows of X whose predicted class equals the label in y."""
    return float(np.mean(predict(network, X) == y))


def make_spirals(n_per_class, n_classes, noise, rng):
    """n_classes interleaved spiral arms in 2-D, sorted by class.

    Returns X with shape (n_per_class * n_classes, 2) and integer labels y with
    shape (n_per_class * n_classes,). noise is the spread of the random angle jitter.
    """
    radius = np.linspace(0.05, 1.0, n_per_class)
    X_parts, y_parts = [], []
    for k in range(n_classes):
        angle = 2 * np.pi * k / n_classes + 6.0 * radius + rng.normal(0.0, noise, n_per_class)
        X_parts.append(np.column_stack([radius * np.cos(angle), radius * np.sin(angle)]))
        y_parts.append(np.full(n_per_class, k))
    return np.concatenate(X_parts), np.concatenate(y_parts)


# From notebook 05: loss functions.

def mse(pred, target):
    """Mean squared error over every entry of two arrays with the same shape."""
    return float(np.mean((pred - target) ** 2))


def binary_cross_entropy(p, y):
    """Mean binary cross-entropy of probabilities p (batch,) against 0/1 labels y (batch,), clipped."""
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def cross_entropy(probs, y):
    """Mean negative log-probability of the true class. probs: (batch, classes), y: (batch,) ints."""
    picked = probs[np.arange(len(y)), y]
    return float(-np.mean(np.log(np.clip(picked, 1e-12, None))))


def cross_entropy_from_logits(Z, y):
    """The same loss computed from logits Z (batch, classes) with the log-sum-exp trick."""
    row_max = Z.max(axis=1, keepdims=True)
    log_sum_exp = row_max[:, 0] + np.log(np.exp(Z - row_max).sum(axis=1))
    return float(np.mean(log_sum_exp - Z[np.arange(len(y)), y]))


# From notebook 06: numerical derivatives and gradient checking.

def numerical_derivative(f, x, h=1e-5):
    """Central-difference estimate of f'(x) for a function of one number."""
    return (f(x + h) - f(x - h)) / (2 * h)


def numerical_gradient(f, array, eps=1e-5):
    """Central-difference gradient of the scalar f(array) with respect to every entry of array.

    Each entry is nudged in place and then restored, so f may also read the array
    through a closure (for example a network's weights). Returns an array shaped like array.
    """
    grad = np.zeros(array.shape)
    for index in np.ndindex(array.shape):
        original = array[index]
        array[index] = original + eps
        loss_plus = f(array)
        array[index] = original - eps
        loss_minus = f(array)
        array[index] = original
        grad[index] = (loss_plus - loss_minus) / (2 * eps)
    return grad


def relative_error(a, b):
    """||a - b|| / (||a|| + ||b||), with a tiny floor on the denominator: 0 when equal, at most 1."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 1 · The plan: store on the way forward, reuse on the way back

Take a network with one hidden layer, such as 2-16-3, and a batch of $n$ examples. The forward pass computes, layer by layer:

| Quantity | Meaning | Shape |
|---|---|---|
| $X = A^{[0]}$ | the inputs | `(n, 2)` |
| $Z^{[1]} = A^{[0]} W^{[1]} + b^{[1]}$ | hidden pre-activations | `(n, 16)` |
| $A^{[1]} = \operatorname{relu}(Z^{[1]})$ | hidden activations | `(n, 16)` |
| $Z^{[2]} = A^{[1]} W^{[2]} + b^{[2]}$ | output logits | `(n, 3)` |
| $P = \operatorname{softmax}(Z^{[2]})$ | probabilities | `(n, 3)` |
| $L = \operatorname{CE}(P, y)$ | the mean loss | a number |

The backward pass runs the chain rule from $L$ back to every weight, reusing the values that the forward pass stored in `cache`: for layer $i$, `cache[i] = (A_prev, Z)`. We write `dW` for $\partial L / \partial W$, and so on. **Every gradient has the same shape as the thing it is the gradient of**: `dW1` has the shape of `W1`, `dZ2` has the shape of `Z2`, and so on. That single rule catches most mistakes.

Backpropagation needs only three local rules, one per kind of step: softmax with cross-entropy at the top, dense layers, and ReLU. You will derive and type each one, then chain them together.

In [ ]:
plan_rng = np.random.default_rng(1)
X_plan = plan_rng.normal(size=(5, 2))          # a batch of n = 5 examples
plan_net = init_network([2, 16, 3], plan_rng)
probs_plan, cache_plan = forward(plan_net, X_plan)
for i, (A_prev, Z) in enumerate(cache_plan):
    print(f"layer {i + 1}: A_prev {A_prev.shape}, W {plan_net[i]['W'].shape}, b {plan_net[i]['b'].shape}, Z {Z.shape}")
print("probs:", probs_plan.shape)

## 2 · Softmax with cross-entropy: $\partial L / \partial Z = (P - Y)/n$

Start at the top. For one example with logits $z_1, \dots, z_K$ and true class $y$, notebook 05 rewrote the loss without probabilities:

$$\ell = -\log p_y = -z_y + \log \sum_j e^{z_j} .$$

Differentiate with respect to one logit $z_k$. The first term gives $-1$ if $k = y$ and $0$ otherwise. For the second term, the chain rule gives

$$\frac{\partial}{\partial z_k} \log \sum_j e^{z_j} = \frac{e^{z_k}}{\sum_j e^{z_j}} = p_k ,$$

which is the softmax itself. Together:

$$\frac{\partial \ell}{\partial z_k} = p_k - [k = y] ,$$

where $[k = y]$ is $1$ for the true class and $0$ otherwise: exactly the one-hot label $Y_k$. The batch loss is the mean $L = \frac{1}{n}\sum_i \ell_i$, and logit $Z_{ik}$ appears only in example $i$'s loss, so

$$\frac{\partial L}{\partial Z} = \frac{P - Y}{n}, \qquad \text{shape } (n, K).$$

**Read it as "predicted minus true".** The true class's logit gets a negative gradient, so gradient descent raises it; every other logit gets a positive gradient and is lowered, in proportion to how much probability it wrongly took. Each row sums to zero, and no entry exceeds $1/n$ in size. The softmax and the logarithm cancel so neatly that this is how every library implements the pair.

The next cell does not use the formula. It measures the gradient numerically, multiplies it by $n$, and prints it beside $P$ and $Y$. Look for the pattern.

In [ ]:
Z_two = np.array([[2.0, 1.0, 0.1],
                  [0.5, 2.5, -1.0]])
y_two = np.array([0, 2])
grad_numerical = numerical_gradient(lambda Z: cross_entropy_from_logits(Z, y_two), Z_two.copy())
print("n × numerical gradient:\n", np.round(len(y_two) * grad_numerical, 6))
print("P = softmax(Z):\n", np.round(softmax(Z_two), 6))
print("Y = one-hot labels:\n", np.eye(3)[y_two])

### ✍️ Your turn: `softmax_cross_entropy_grad`

Write `softmax_cross_entropy_grad(probs, y)`:

- **inputs:** `probs`, the softmax output of shape `(n, K)`, and `y`, integer labels of shape `(n,)`;
- **output:** $\partial L / \partial Z = (P - Y)/n$, shape `(n, K)`, where $Y$ is the one-hot version of `y` and $n$ is the batch size.

Hints: `np.eye(K)[y]` turns labels into one-hot rows, with `K = probs.shape[1]`; `n = len(y)`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def softmax_cross_entropy_grad_reference(probs, y):
    return (probs - np.eye(probs.shape[1])[y]) / len(y)

In [ ]:
def softmax_cross_entropy_grad(probs, y):
    """dL/dZ for softmax + mean cross-entropy. probs: (n, K), y: (n,) ints. Returns (P - Y) / n, shape (n, K)."""
    # ✍️ Your code here (one or two lines)
    raise NotImplementedError

In [ ]:
grad_rng = np.random.default_rng(3)
softmax_cross_entropy_grad = practice("softmax_cross_entropy_grad", softmax_cross_entropy_grad,
                                      softmax_cross_entropy_grad_reference, cases=[
    (softmax(Z_two), y_two),
    (softmax(grad_rng.normal(size=(4, 3))), np.array([2, 0, 1, 1])),
    (softmax(np.array([[0.3, -0.3]])), np.array([1])),     # edge case: one example, two classes
])

**Expected behaviour.** For the two examples above, $n \times$ the gradient is `[[-0.341, 0.242, 0.099], [0.116, 0.858, -0.974]]` to three decimals, exactly $P - Y$: the true-class entries (column 0 in row 1, column 2 in row 2) are negative, and each row sums to zero. Notice that the second example, which put only $0.026$ on its true class, gets a much larger push than the first.

## 3 · A dense layer: $dW = A^\top dZ$, $db = \sum_i dZ_i$, $dA = dZ\,W^\top$

A dense layer computes $Z = A W + b$, where $A$ is the layer's input `A_prev` with shape `(n, n_in)`, $W$ is `(n_in, n_out)` and $b$ is `(n_out,)`. Entry by entry,

$$Z_{ik} = \sum_j A_{ij} W_{jk} + b_k .$$

Suppose the layer above has already given us $dZ = \partial L / \partial Z$, shape `(n, n_out)`. A weight affects the loss through every entry of $Z$ that it touches, so the chain rule **adds** those contributions.

- **Weights.** $W_{jk}$ appears in $Z_{ik}$ for every example $i$, with local derivative $\partial Z_{ik} / \partial W_{jk} = A_{ij}$:
  $$\frac{\partial L}{\partial W_{jk}} = \sum_i dZ_{ik}\, A_{ij} = (A^\top dZ)_{jk} \quad\Rightarrow\quad dW = A^\top dZ, \ \text{shape } (n_\text{in}, n_\text{out}).$$
- **Biases.** $b_k$ is added to $Z_{ik}$ in every row, with local derivative $1$:
  $$\frac{\partial L}{\partial b_k} = \sum_i dZ_{ik} \quad\Rightarrow\quad db = \textstyle\sum_i dZ_{i,:}, \ \text{shape } (n_\text{out},).$$
- **Inputs.** $A_{ij}$ appears in $Z_{ik}$ for every output unit $k$, with local derivative $W_{jk}$:
  $$\frac{\partial L}{\partial A_{ij}} = \sum_k dZ_{ik}\, W_{jk} = (dZ\, W^\top)_{ij} \quad\Rightarrow\quad dA_\text{prev} = dZ\, W^\top, \ \text{shape } (n, n_\text{in}).$$

The input gradient $dA_\text{prev}$ is what the layer passes down to the layer below it: that is the "propagation" in backpropagation.

**Two things to notice.** There is no division by $n$ here: the $1/n$ is already inside $dZ$, because it came from the mean in the loss. And you can rebuild each formula from shapes alone: `dW` must be `(n_in, n_out)`, and the only product of `A_prev` `(n, n_in)` and `dZ` `(n, n_out)` with that shape is `A_prev.T @ dZ`.

In [ ]:
shape_rng = np.random.default_rng(4)
A_demo, W_demo, dZ_demo = shape_rng.normal(size=(5, 4)), shape_rng.normal(size=(4, 3)), shape_rng.normal(size=(5, 3))
print("A_prev.T @ dZ has shape", (A_demo.T @ dZ_demo).shape, "and W has shape", W_demo.shape)
try:
    A_demo @ dZ_demo
except ValueError as error:
    print("A_prev @ dZ fails:", str(error)[:70], "...")

### ✍️ Your turn: `dense_backward`

Write `dense_backward(dZ, A_prev, W)`:

- **inputs:** `dZ`, the gradient of the loss with respect to the layer's output, shape `(n, n_out)`; `A_prev`, the layer's input, shape `(n, n_in)`; `W`, the layer's weights, shape `(n_in, n_out)`;
- **output:** the tuple `(dW, db, dA_prev)` with shapes `(n_in, n_out)`, `(n_out,)` and `(n, n_in)`.

Hints: one line each. `dZ.sum(axis=0)` adds over the batch; `.T` transposes.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below.

In [ ]:
def dense_backward_reference(dZ, A_prev, W):
    return A_prev.T @ dZ, dZ.sum(axis=0), dZ @ W.T

In [ ]:
def dense_backward(dZ, A_prev, W):
    """Gradients of Z = A_prev @ W + b. dZ: (n, n_out), A_prev: (n, n_in), W: (n_in, n_out).

    Returns (dW, db, dA_prev) with shapes (n_in, n_out), (n_out,) and (n, n_in).
    """
    # ✍️ Your code here (three short lines)
    raise NotImplementedError

In [ ]:
dense_rng = np.random.default_rng(5)
dense_backward = practice("dense_backward", dense_backward, dense_backward_reference, cases=[
    (dZ_demo, A_demo, W_demo),
    (dense_rng.normal(size=(3, 2)), dense_rng.normal(size=(3, 5)), dense_rng.normal(size=(5, 2))),
    (dense_rng.normal(size=(1, 3)), dense_rng.normal(size=(1, 4)), dense_rng.normal(size=(4, 3))),  # a batch of one
])

**Expected behaviour.** The three results have exactly the shapes of `W`, `b` and `A_prev`. With a batch of one, `dW` is the **outer product** of that example's input and its `dZ`: every weight $W_{jk}$ connects input $j$ to output $k$, so its gradient is "how active the input was" times "how much the output should change".

## 4 · ReLU: let the gradient through where the unit was on

A hidden layer applies $A = \operatorname{relu}(Z)$ entry by entry, and each $A_{ij}$ depends only on $Z_{ij}$, with slope $1$ if $Z_{ij} > 0$ and $0$ otherwise. So the chain rule needs no sum:

$$dZ = dA \odot \operatorname{relu\_grad}(Z),$$

where $\odot$ multiplies entry by entry. Wherever the unit was switched on, the gradient passes through unchanged; wherever it was off, the gradient is blocked. At exactly $Z = 0$ we use the convention that the slope is $0$, as `relu_grad` does.

One consequence: a unit that is off for **every** example in the batch gets no gradient at all and cannot learn from that batch. If it stays off for all inputs, it is a "dead" ReLU. You will meet one in notebook 08.

**Shapes.** `dA`, `Z` and the result all have the same shape, `(n, units)`.

In [ ]:
Z_mask = np.array([[1.5, -0.3, 0.0],
                   [-2.0, 0.7, 4.0]])
print("relu_grad(Z), the mask:\n", relu_grad(Z_mask))

### ✍️ Your turn: `relu_backward`

Write `relu_backward(dA, Z)`:

- **inputs:** `dA`, the gradient with respect to the ReLU's output, and `Z`, the ReLU's input from the forward pass, both of shape `(n, units)`;
- **output:** `dZ`, the gradient with respect to `Z`, of the same shape.

Hint: multiply `dA` by `relu_grad(Z)`. Do not change `dA` in place, because the caller may still need it.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below.

In [ ]:
def relu_backward_reference(dA, Z):
    return dA * relu_grad(Z)

In [ ]:
def relu_backward(dA, Z):
    """dL/dZ for A = relu(Z). dA and Z: (n, units). Returns dZ of the same shape."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
relu_rng = np.random.default_rng(6)
relu_backward = practice("relu_backward", relu_backward, relu_backward_reference, cases=[
    (np.ones((2, 3)), Z_mask),                          # Z contains an exact 0, which must block the gradient
    (relu_rng.normal(size=(4, 5)), relu_rng.normal(size=(4, 5))),
])

**Expected behaviour.** With `dA` full of ones the result is exactly the mask: `[[1, 0, 0], [0, 1, 1]]`. The entry where $Z = 0$ is blocked.

## 5 · The full backward pass

Now chain the pieces from the output layer down to the first layer. In pseudo-code:

```
dZ = softmax_cross_entropy_grad(probs, y)          # the output layer's dZ
for i = last layer, ..., first layer:
    A_prev, Z = cache[i]
    if layer i is a hidden layer:
        dZ = relu_backward(dA, Z)                  # dA came from the layer above
    dW, db, dA = dense_backward(dZ, A_prev, network[i]["W"])
    grads[i] = {"W": dW, "b": db}
```

The output layer has no ReLU, because softmax is already included in its `dZ`. At the bottom, `dA` is the gradient with respect to the input `X`; training ignores it, but it is a free by-product (Exercise 3 uses it). The `grads` list has the same structure as `network`, so `grads[i]["W"]` has the shape of `network[i]["W"]`.

### ✍️ Your turn: `backward`

Write `backward(network, cache, probs, y)`:

- **inputs:** `network`, a list of layers `{"W": (n_in, n_out), "b": (n_out,)}`; `cache` and `probs`, returned by `forward(network, X)`; `y`, the integer labels of shape `(n,)`;
- **output:** `grads`, a list with one dict `{"W": dW, "b": db}` per layer, in the same order as `network`.

Use the three functions you have just written. Hints: create `grads = [None] * len(network)`; loop with `for i in reversed(range(len(network))):`; a layer is hidden when `i < len(network) - 1`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below.

In [ ]:
def backward_reference(network, cache, probs, y):
    grads = [None] * len(network)
    dZ = (probs - np.eye(probs.shape[1])[y]) / len(y)
    for i in reversed(range(len(network))):
        A_prev, Z = cache[i]
        if i < len(network) - 1:
            dZ = dA * relu_grad(Z)
        dW, db, dA = A_prev.T @ dZ, dZ.sum(axis=0), dZ @ network[i]["W"].T
        grads[i] = {"W": dW, "b": db}
    return grads

In [ ]:
def backward(network, cache, probs, y):
    """Gradients of the mean cross-entropy for every layer.

    network: list of {"W", "b"}; cache, probs: from forward(network, X); y: (n,) ints.
    Returns a list of {"W": dW, "b": db}, one per layer, shaped like the network.
    """
    # ✍️ Your code here (about eight lines)
    raise NotImplementedError

In [ ]:
def backward_case(sizes, n, seed):
    # Build a small network and a batch, run forward, and return the arguments for backward.
    case_rng = np.random.default_rng(seed)
    case_net = init_network(sizes, case_rng)
    X_case = case_rng.normal(size=(n, sizes[0]))
    y_case = case_rng.integers(0, sizes[-1], size=n)
    case_probs, case_cache = forward(case_net, X_case)
    return case_net, case_cache, case_probs, y_case

backward = practice("backward", backward, backward_reference, cases=[
    backward_case([2, 4, 3], n=5, seed=0),
    backward_case([2, 5, 4, 3], n=6, seed=1),     # two hidden layers
    backward_case([2, 3], n=4, seed=2),           # edge case: no hidden layer at all
])

**Expected behaviour.** ✅ on all three networks, including the one with no hidden layer, where the loop runs once and never touches a ReLU. If you see ❌ for the two-hidden-layer network only, check which `Z` you mask with: it must come from `cache[i]` for the layer you are currently in.

## 6 · Gradient-check every parameter

Matching the reference on three cases is encouraging, but the real test is independent of any reference: compare every gradient with `numerical_gradient` from notebook 06. The network below has two hidden layers, 2-5-4-3, so every rule you wrote is exercised, and its biases are set to small random values so that they are tested in a typical state rather than at zero.

`network_loss` ignores its argument and runs a full forward pass. `numerical_gradient` nudges each weight **inside** the network in place, so every nudge shows up in the loss.

In [ ]:
check_rng = np.random.default_rng(7)
X_check, y_check = make_spirals(4, 3, 0.2, check_rng)          # 12 examples
check_net = init_network([2, 5, 4, 3], check_rng)
for layer in check_net:
    layer["b"] = check_rng.normal(0.0, 0.1, size=layer["b"].shape)

def network_loss(_):
    check_probs_now, _cache = forward(check_net, X_check)
    return cross_entropy(check_probs_now, y_check)

check_probs, check_cache = forward(check_net, X_check)
check_grads = backward(check_net, check_cache, check_probs, y_check)
numeric_grads = [{name: numerical_gradient(network_loss, layer[name]) for name in ("W", "b")} for layer in check_net]

print(f"{'parameter':<10} {'shape':<9} {'relative error':>15}")
check_errors = []
for i, layer in enumerate(check_net):
    for name in ("W", "b"):
        error = relative_error(check_grads[i][name], numeric_grads[i][name])
        check_errors.append(error)
        print(f"{name}{i + 1:<9} {str(layer[name].shape):<9} {error:>15.1e}")
closest_to_kink = min(float(np.abs(Z).min()) for _, Z in check_cache[:-1])
print(f"\nlargest relative error: {max(check_errors):.1e}")
print(f"smallest |Z| in the hidden layers: {closest_to_kink:.1e}, far larger than the nudge of 1e-5")

### Why the check works

- **The two methods are independent.** The numerical gradient uses nothing but forward passes; it never runs a line of your backward code. If the two agree on every one of the 54 parameters, the backward pass computes the gradient of the function that the forward pass defines. (The check cannot tell you whether the forward pass is the function you *intended*. That is what the tests of earlier notebooks are for.)
- **Correct code agrees almost perfectly.** The central difference has a truncation error proportional to $\varepsilon^2 = 10^{-10}$, and rounding adds roughly $10^{-16} / \varepsilon = 10^{-11}$. So a correct backward pass matches to a relative error of about $10^{-9}$ or better, as in the table.
- **Wrong code disagrees by a lot.** A bug drops or adds a whole term or factor, such as a missing $1/n$, a missing mask or a wrong transpose, so the relative error is typically between $10^{-3}$ and $1$. Many orders of magnitude separate the two cases, so a threshold of $10^{-6}$ is safe.
- **It points at the bug.** Each parameter array is checked separately, so a failure tells you which layer to look at, as the experiment below shows.

Three conditions make the check reliable. Use `float64`. Stay away from kinks: a nudge that pushes a hidden $Z$ across $0$ changes ReLU's slope, and the last printed line confirms that every hidden $|Z|$ is far larger than the nudge. And keep the network and batch **small**, because the numerical gradient costs two forward passes per parameter.

### 🧪 Try it: break a gradient on purpose

`broken_backward` is a copy of the backward pass with one deliberate mistake, chosen by `bug`. Your own `backward` is untouched. Run the cell, then change `bug` and run it again. Before each run, predict which rows of the table will fail.

In [ ]:
def broken_backward(network, cache, probs, y, bug):
    grads = [None] * len(network)
    dZ = probs - np.eye(probs.shape[1])[y]
    if bug != "forget the 1/n":
        dZ = dZ / len(y)
    for i in reversed(range(len(network))):
        A_prev, Z = cache[i]
        if i < len(network) - 1:
            dZ = dA if bug == "forget the ReLU mask" else dA * relu_grad(Z)
        db = dZ.mean(axis=0) if bug == "average the bias gradient" else dZ.sum(axis=0)
        grads[i] = {"W": A_prev.T @ dZ, "b": db}
        dA = dZ @ network[i]["W"].T
    return grads

bug = "forget the ReLU mask"     # try "forget the 1/n" and "average the bias gradient", then run the cell again
broken_grads = broken_backward(check_net, check_cache, check_probs, y_check, bug)
print(f"bug: {bug}\n")
for i in range(len(check_net)):
    for name in ("W", "b"):
        error = relative_error(broken_grads[i][name], numeric_grads[i][name])
        verdict = "✅ agrees" if error < 1e-6 else "❌ caught"
        print(f"{name}{i + 1:<4} relative error {error:9.1e}   {verdict}")

**What to notice.**

- *Forget the ReLU mask:* the output layer (`W3`, `b3`) still agrees, because no mask is involved there, but both hidden layers are caught. The bug lives between layer 3 and the layers below it.
- *Forget the 1/n:* every gradient is $n = 12$ times too large, so every row fails with the same relative error, $(12 - 1)/(12 + 1) \approx 0.85$. The same error everywhere suggests a scale factor.
- *Average the bias gradient:* only the `b` rows fail, all with about $0.85$ again, because `mean` divides by $n$ a second time. Every `W` row agrees.

Each bug leaves its own fingerprint in the table, so reading it tells you where to look.

### How much faster is backpropagation?

For the 2-64-3 network of the next notebook, the numerical gradient needs two forward passes for each of its 387 parameters. Backpropagation needs one forward and one backward pass in total. The cell times both on 300 examples, averaging the fast one over 20 runs. The exact times depend on your computer, but the gap is always large.

In [ ]:
import time

speed_rng = np.random.default_rng(8)
X_speed, y_speed = make_spirals(100, 3, 0.2, speed_rng)
speed_net = init_network([2, 64, 3], speed_rng)

start = time.perf_counter()
for _ in range(20):
    speed_probs, speed_cache = forward(speed_net, X_speed)
    backward(speed_net, speed_cache, speed_probs, y_speed)
backprop_seconds = (time.perf_counter() - start) / 20

start = time.perf_counter()
for layer in speed_net:
    for name in ("W", "b"):
        numerical_gradient(lambda _: cross_entropy(forward(speed_net, X_speed)[0], y_speed), layer[name])
numerical_seconds = time.perf_counter() - start

n_parameters = sum(layer["W"].size + layer["b"].size for layer in speed_net)
print(f"{n_parameters} parameters: backpropagation {backprop_seconds * 1000:.2f} ms, "
      f"numerical gradient {numerical_seconds * 1000:.0f} ms, about {numerical_seconds / backprop_seconds:.0f} times slower")

On a laptop running CPython, backpropagation takes about a tenth of a millisecond here and the numerical gradient is roughly 400 times slower; in the browser both are slower, but the gap remains. The ratio grows with the size of the network: the numerical gradient needs two forward passes per parameter, while backpropagation needs a single backward pass however many parameters there are. For a network with millions of weights, numerical gradients are out of the question for training, but a small gradient check remains the best test of the code that training relies on.

## 7 · Put it together: check it, then use it

The final cell takes a fresh network and a fresh batch, gradient-checks **every** parameter, and asserts that all relative errors are below $10^{-6}$. Then it uses the gradients for what they are for: 300 steps of plain gradient descent, $W \leftarrow W - \eta\, dW$ and $b \leftarrow b - \eta\, db$, on 150 spiral points. The loss must fall steadily. Notebook 08 turns this loop into a proper training procedure.

In [ ]:
final_rng = np.random.default_rng(2025)
X_small, y_small = make_spirals(3, 3, 0.2, final_rng)           # 9 examples for the gradient check
small_net = init_network([2, 6, 5, 3], final_rng)
small_probs, small_cache = forward(small_net, X_small)
small_grads = backward(small_net, small_cache, small_probs, y_small)
worst_error = 0.0
for i, layer in enumerate(small_net):
    for name in ("W", "b"):
        numeric = numerical_gradient(lambda _: cross_entropy(forward(small_net, X_small)[0], y_small), layer[name])
        worst_error = max(worst_error, relative_error(small_grads[i][name], numeric))
print(f"gradient check of {sum(l['W'].size + l['b'].size for l in small_net)} parameters: worst relative error {worst_error:.1e}")

X_train, y_train = make_spirals(50, 3, 0.2, final_rng)          # 150 examples for training
train_net = init_network([2, 32, 3], final_rng)
learning_rate = 1.0
losses = []
for step in range(300):
    train_probs, train_cache = forward(train_net, X_train)
    losses.append(cross_entropy(train_probs, y_train))
    train_grads = backward(train_net, train_cache, train_probs, y_train)
    for layer, grad in zip(train_net, train_grads):
        layer["W"] -= learning_rate * grad["W"]
        layer["b"] -= learning_rate * grad["b"]
final_accuracy = accuracy(train_net, X_train, y_train)
print(f"loss: {losses[0]:.4f} at the start, {losses[-1]:.4f} after 300 steps; training accuracy {final_accuracy:.3f}")

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot(losses, color="#2a78d6", linewidth=2)
ax.set_xlabel("gradient descent step")
ax.set_ylabel("cross-entropy loss")
ax.grid(alpha=0.3)
ax.set_title("Your gradients at work: the loss goes down")
plt.show()

assert worst_error < 1e-6, "every gradient should match the numerical gradient"
assert losses[-1] < 0.6 * losses[0], "300 steps of gradient descent should cut the loss substantially"
assert final_accuracy > 0.6, "the network should classify most training points correctly"
print("✅ All checks passed.")

**What to notice.** All parameters pass the check, and the loss falls steadily from about $\log 3$. The network is learning purely from the gradients that your `backward` computed. That is backpropagation: the chain rule, organised so that each layer reuses the work of the layer above it.

## Exercises

### Exercise 1 · Backward through a sigmoid

The sigmoid's slope is $\sigma(z)\,(1 - \sigma(z))$. Write `sigmoid_backward(dA, Z)`, the sigmoid counterpart of `relu_backward`. The check cell also compares your function with `numerical_gradient` on a small loss.

In [ ]:
def sigmoid_backward_reference(dA, Z):
    s = sigmoid(Z)
    return dA * s * (1.0 - s)

In [ ]:
def sigmoid_backward(dA, Z):
    """dL/dZ for A = sigmoid(Z). dA and Z: (n, units). Returns dZ of the same shape."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
sig_rng = np.random.default_rng(9)
Z_sig, weights_sig = sig_rng.normal(size=(3, 4)), sig_rng.normal(size=(3, 4))
sigmoid_backward = practice("sigmoid_backward", sigmoid_backward, sigmoid_backward_reference, cases=[
    (weights_sig, Z_sig),
    (np.ones((1, 3)), np.array([[-20.0, 0.0, 20.0]])),   # saturated units pass almost nothing back
])
# The loss sum(weights * sigmoid(Z)) has dL/dA = weights, so its gradient with respect to Z is sigmoid_backward(weights, Z).
numeric_sig = numerical_gradient(lambda Z: float(np.sum(weights_sig * sigmoid(Z))), Z_sig.copy())
print(f"gradient check: relative error {relative_error(sigmoid_backward(weights_sig, Z_sig), numeric_sig):.1e}")

### Exercise 2 · The output gradient for regression

A regression network has no softmax: its output layer's `Z` is the prediction itself, and the loss is `mse(pred, target)`, the mean over **every** entry. Derive $\partial L / \partial \text{pred}$ and write `mse_output_grad(pred, target)`. Hint: count the entries the mean divides by; it is `pred.size`, not just the batch size.

In [ ]:
def mse_output_grad_reference(pred, target):
    return 2.0 * (pred - target) / pred.size

In [ ]:
def mse_output_grad(pred, target):
    """dL/dpred for L = mse(pred, target). pred and target: the same shape. Returns an array of that shape."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
mse_rng = np.random.default_rng(10)
pred_reg, target_reg = mse_rng.normal(size=(4, 2)), mse_rng.normal(size=(4, 2))
mse_output_grad = practice("mse_output_grad", mse_output_grad, mse_output_grad_reference, cases=[
    (pred_reg, target_reg),
    (np.array([[1.0], [2.0], [3.0]]), np.array([[1.0], [2.5], [2.0]])),
])
numeric_mse = numerical_gradient(lambda P: mse(P, target_reg), pred_reg.copy())
print(f"gradient check: relative error {relative_error(mse_output_grad(pred_reg, target_reg), numeric_mse):.1e}")

### Exercise 3 · The gradient with respect to the inputs

The backward pass also produces $\partial L / \partial X$, the gradient with respect to the inputs, which `backward` throws away. It tells you which small change to an input would increase the loss fastest: the idea behind saliency maps and adversarial examples. Write `input_gradient(network, X, y)`, which runs the forward pass and the full backward pass and returns `dX` with the shape of `X`.

In [ ]:
def input_gradient_reference(network, X, y):
    probs, cache = forward(network, X)
    dZ = (probs - np.eye(probs.shape[1])[y]) / len(y)
    for i in reversed(range(len(network))):
        A_prev, Z = cache[i]
        if i < len(network) - 1:
            dZ = dA * relu_grad(Z)
        dA = dZ @ network[i]["W"].T
    return dA

In [ ]:
def input_gradient(network, X, y):
    """dL/dX for the mean cross-entropy of the network on (X, y). Returns an array shaped like X."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
input_gradient = practice("input_gradient", input_gradient, input_gradient_reference, cases=[
    (check_net, X_check, y_check),
    (small_net, X_small, y_small),
])
numeric_dX = numerical_gradient(lambda X: cross_entropy(forward(check_net, X)[0], y_check), X_check.copy())
print(f"gradient check: relative error {relative_error(input_gradient(check_net, X_check, y_check), numeric_dX):.1e}")

### Exercise 4 · Open questions

1. The speed comparison above used 387 parameters. Roughly how many forward passes would one numerical gradient need for a network with one million parameters, and how does that compare with backpropagation?
2. `backward` divides by $n$ only once, in `softmax_cross_entropy_grad`. Explain why the biases' gradient uses a **sum** over the batch rather than a mean.
3. The ReLU mask can be computed from $A$ instead of $Z$, since $A > 0$ exactly when $Z > 0$. Would the same trick work for a sigmoid? Why not?

## Common errors and debugging

| Symptom | Likely cause | Fix |
|---|---|---|
| `ValueError: matmul: Input operand 1 has a mismatch in its core dimension ...` | a missing or extra transpose | write down the shapes; `dW = A_prev.T @ dZ` and `dA_prev = dZ @ W.T` are the only arrangements that fit |
| `db` has shape `(n,)` instead of `(n_out,)` | the sum ran over the wrong axis | sum over the batch: `dZ.sum(axis=0)` |
| every check fails with the same relative error | a scale factor: $1/n$ was forgotten or applied twice | divide by $n$ once, in the output gradient only |
| only the hidden layers fail | the ReLU mask is missing or comes from the wrong layer | mask with `Z` from `cache[i]` of the layer you are in |
| only the `b` rows fail | `mean` instead of `sum` for the bias gradient | `db = dZ.sum(axis=0)` |
| results change between runs of the same cell | the backward pass modified a cached array in place, for example with `dZ[Z <= 0] = 0` on a shared array | build new arrays: `dZ = dA * relu_grad(Z)` |
| one or two entries fail by a little | a nudge crossed a ReLU kink | check at another random point, or print the smallest hidden $\lvert Z \rvert$ |

## Summary

- Backpropagation applies the chain rule from the loss back to every parameter, reusing the forward pass's `cache`. It costs about one extra forward pass, not two per parameter.
- **Softmax with cross-entropy:** $\partial L / \partial Z = (P - Y)/n$, "predicted minus true".
- **Dense layer:** $dW = A_\text{prev}^\top dZ$, $db = \sum_i dZ_i$ and $dA_\text{prev} = dZ\, W^\top$. Every gradient has the shape of its parameter, and shapes alone fix the transposes.
- **ReLU:** $dZ = dA \odot [Z > 0]$; switched-off units block the gradient.
- **Gradient checking** compares every parameter's gradient with a numerical one. Correct code gives relative errors around $10^{-9}$ or less, bugs give $10^{-3}$ to $1$, and the pattern of failures points at the bug.

You built `softmax_cross_entropy_grad`, `dense_backward`, `relu_backward` and `backward`. The next notebook uses them under these names.

## Next

You can now compute exact gradients quickly. In **08 · Train with gradient descent** (`08_train_with_gradient_descent.ipynb`) you turn them into a complete training loop with mini-batches, epochs and a validation set, explore learning rates that are too small and too large, and train a network that untangles the spirals.